# 🔱 Shiv AI Voice Cloning v2.0
**Owner: Shri Ram Nag | PAISAWALA Channel**

### Features:
- 🎙️ **Voice Clone** — Reference audio se aawaz clone
- 🎛️ **Voice Design** — Speed, Pitch, Energy, Pause controls + Presets
- 🔤 **Simple TTS** — Seedha text se audio
- 📋 **Instruct Mode** — Instructions se voice style control

> ⚡ **IMPORTANT:** Runtime → Change runtime type → **T4 GPU** select karein pehle!

In [ ]:
# ✅ STEP 1: GPU Check
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    raise RuntimeError('❌ GPU nahi mila! Runtime → Change runtime type → T4 GPU select karein!')

In [ ]:
# ✅ STEP 2: Dependencies Install
!pip install -q gradio huggingface_hub transformers accelerate scipy numpy
print('✅ Dependencies installed!')

In [ ]:
# ✅ STEP 3: Model Download from HuggingFace
import os
from huggingface_hub import snapshot_download

REPO_ID = 'Shriramnag/Shiv-AI-Voice-Cloning'
LOCAL_DIR = './Shiv-AI-Voice-Cloning'

if not os.path.exists(LOCAL_DIR) or not os.listdir(LOCAL_DIR):
    print(f'📥 Downloading: {REPO_ID}')
    print('⏳ Please wait... (~3.27 GB total)')
    snapshot_download(
        repo_id=REPO_ID,
        local_dir=LOCAL_DIR,
        local_dir_use_symlinks=False
    )
    print('✅ Download complete!')
else:
    print('✅ Model already downloaded!')

print('\n📂 Files:')
for f in sorted(os.listdir(LOCAL_DIR)):
    size = os.path.getsize(os.path.join(LOCAL_DIR, f)) / 1e6
    print(f'  {f:35s} {size:.2f} MB')

In [ ]:
# ✅ STEP 4: Imports & Model Load
import sys, re, uuid, logging
import numpy as np
import torch
import gradio as gr

MODEL_LOCAL_PATH = './Shiv-AI-Voice-Cloning'
sys.path.insert(0, MODEL_LOCAL_PATH)

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name

try:
    from subtitle import subtitle_maker
    from subtitle import LANGUAGE_CODE as WHISPER_LANGUAGE_CODE
except ImportError:
    WHISPER_LANGUAGE_CODE = None

print('🔱 Loading Shiv AI Model...')
model = OmniVoice.from_pretrained(
    MODEL_LOCAL_PATH,
    device_map='cuda',
    dtype=torch.float16,
    load_asr=False,
)
sampling_rate = model.sampling_rate
print(f'✅ Model loaded! Sample rate: {sampling_rate} Hz')

In [ ]:
# ✅ STEP 5: Helper Functions & Constants
os.makedirs('./Shiv_Audio', exist_ok=True)

EVENT_TAGS = ['[laughter]','[sigh]','[confirmation-en]','[question-en]','[surprise-wa]','[dissatisfaction-hnn]']
LANG_CHOICES = ['Auto'] + sorted(lang_display_name(n) for n in LANG_NAMES)

INSERT_TAG_JS = """
(tag_val, current_text) => {
    const textarea = document.querySelector('.shiv-textbox textarea');
    if (!textarea) return current_text + ' ' + tag_val;
    const start = textarea.selectionStart;
    const end = textarea.selectionEnd;
    return current_text.slice(0, start) + ' ' + tag_val + ' ' + current_text.slice(end);
}
"""

INSTRUCT_EXAMPLES = [
    'Speak slowly and clearly with a calm, deep voice',
    'Speak with excitement and high energy',
    'Speak softly like a bedtime story',
    'Speak like a news anchor, formal and clear',
    'Speak in a sad, emotional tone',
    'Speak fast and enthusiastically like a sports commentator',
    'धीरे और शांत आवाज़ में बोलें',
    'जोश और उत्साह के साथ बोलें',
]

def make_gen_config(num_step=32, guidance_scale=2.0):
    return OmniVoiceGenerationConfig(
        num_step=num_step, guidance_scale=guidance_scale,
        denoise=True, preprocess_prompt=True, postprocess_output=True,
    )

def gen_voice_clone(text, language, ref_audio, ref_text=''):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    if not ref_audio: return None, '⚠️ Reference audio upload karein'
    try:
        kw = dict(
            text=text.strip(),
            language=language if language != 'Auto' else None,
            generation_config=make_gen_config(),
            voice_clone_prompt=model.create_voice_clone_prompt(
                ref_audio=ref_audio,
                ref_text=ref_text.strip() if ref_text else None
            )
        )
        audio = model.generate(**kw)
        return (sampling_rate, (audio[0] * 32767).astype(np.int16)), '✅ Voice Clone सफल!'
    except Exception as e:
        return None, f'❌ {e}'

def gen_voice_design(text, language, speed, pitch, energy, pause, style_instruct):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        kw = dict(
            text=text.strip(),
            language=language if language != 'Auto' else None,
            generation_config=OmniVoiceGenerationConfig(
                num_step=32, guidance_scale=2.5, denoise=True,
                preprocess_prompt=True, postprocess_output=True,
                speed=speed, pitch=pitch, energy=energy,
            ),
        )
        if style_instruct: kw['instruct'] = style_instruct
        audio = model.generate(**kw)
        return (sampling_rate, (audio[0] * 32767).astype(np.int16)), f'✅ Voice Design applied! speed={speed} pitch={pitch} energy={energy}'
    except Exception as e:
        try:
            kw2 = dict(text=text.strip(), language=language if language!='Auto' else None, generation_config=make_gen_config())
            if style_instruct: kw2['instruct'] = style_instruct
            audio = model.generate(**kw2)
            return (sampling_rate, (audio[0]*32767).astype(np.int16)), f'✅ Basic mode — {e}'
        except Exception as e2:
            return None, f'❌ {e2}'

def gen_tts(text, language, num_step, guidance_scale):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        audio = model.generate(text=text.strip(), language=language if language!='Auto' else None,
                                generation_config=make_gen_config(int(num_step), guidance_scale))
        return (sampling_rate, (audio[0]*32767).astype(np.int16)), '✅ TTS सफल!'
    except Exception as e:
        return None, f'❌ {e}'

def gen_instruct(text, language, instruct_prompt):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    if not instruct_prompt or not instruct_prompt.strip(): return None, '⚠️ Instruction likhein'
    try:
        audio = model.generate(text=text.strip(), language=language if language!='Auto' else None,
                                generation_config=make_gen_config(guidance_scale=3.0),
                                instruct=instruct_prompt.strip())
        return (sampling_rate, (audio[0]*32767).astype(np.int16)), f'✅ Instruct: {instruct_prompt[:40]}'
    except Exception as e:
        return None, f'❌ {e}'

print('✅ All functions ready!')

In [ ]:
# ✅ STEP 6: Launch Shiv AI UI (All 4 Tabs)
theme = gr.themes.Soft(primary_hue='orange', font=['Inter','Arial','sans-serif'])
css = """
.gradio-container {max-width:100%!important;}
footer {visibility:hidden!important;}
.shiv-header {text-align:center;margin:20px auto;padding:15px;border-bottom:3px solid #ff6600;}
.tag-btn {background:#fff3e0!important;border:1px solid #ffcc80!important;color:#e65100!important;font-size:0.8em!important;}
.gen-btn {font-size:1.1em!important;padding:12px!important;}
"""

with gr.Blocks(theme=theme, css=css, title='🔱 Shiv AI Voice Cloning') as demo:
    gr.HTML("""
        <div class='shiv-header'>
            <h1 style='font-size:2.5em;color:#ff6600;margin-bottom:4px;'>🔱 Shiv AI Voice Cloning</h1>
            <p style='font-size:1.1em;color:#444;'><b>Owner: Shri Ram Nag</b> | PAISAWALA 🎬</p>
            <p style='font-size:0.85em;color:#888;'>Model: Shriramnag/Shiv-AI-Voice-Cloning | 646 Languages</p>
        </div>
    """)

    with gr.Tabs():

        # TAB 1: Voice Clone
        with gr.TabItem('🎙️ Voice Clone'):
            with gr.Row():
                with gr.Column():
                    vc_text = gr.Textbox(label='📝 Text', lines=5, elem_classes='shiv-textbox', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b.click(fn=None, inputs=[b, vc_text], outputs=vc_text, js=INSERT_TAG_JS)
                    vc_lang = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    vc_ref = gr.Audio(label='🎤 Reference Audio', type='filepath')
                    vc_ref_text = gr.Textbox(label='📄 Reference Transcript (optional)', lines=2)
                    vc_btn = gr.Button('🔱 Clone Voice', variant='primary', size='lg')
                with gr.Column():
                    vc_out = gr.Audio(label='🔊 Output', type='numpy')
                    vc_status = gr.Textbox(label='Status', interactive=False)
            vc_btn.click(gen_voice_clone, [vc_text, vc_lang, vc_ref, vc_ref_text], [vc_out, vc_status])

        # TAB 2: Voice Design
        with gr.TabItem('🎛️ Voice Design'):
            with gr.Row():
                with gr.Column():
                    vd_text = gr.Textbox(label='📝 Text', lines=5, elem_classes='shiv-textbox', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b2 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b2.click(fn=None, inputs=[b2, vd_text], outputs=vd_text, js=INSERT_TAG_JS)
                    vd_lang = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    vd_speed  = gr.Slider(label='⚡ Speed',  minimum=0.5, maximum=2.0, value=1.0, step=0.05, info='0.5=Slow | 1.0=Normal | 2.0=Fast')
                    vd_pitch  = gr.Slider(label='🎵 Pitch',  minimum=-12,  maximum=12,  value=0,   step=1,    info='-12=Deep | 0=Normal | +12=High')
                    vd_energy = gr.Slider(label='💪 Energy', minimum=0.3, maximum=2.0, value=1.0, step=0.05, info='0.3=Soft | 1.0=Normal | 2.0=Loud')
                    vd_pause  = gr.Slider(label='⏸️ Pause',  minimum=0.0, maximum=1.0, value=0.0, step=0.1,  info='0=No pause | 1=Long pauses')
                    vd_style  = gr.Textbox(label='✍️ Style Instruction (optional)', lines=2)
                    with gr.Row():
                        p_calm    = gr.Button('😌 Calm',        size='sm')
                        p_excited = gr.Button('🔥 Excited',     size='sm')
                        p_news    = gr.Button('📺 News Anchor', size='sm')
                        p_story   = gr.Button('📖 Story',       size='sm')
                    vd_btn = gr.Button('🎛️ Design & Generate', variant='primary', size='lg')
                with gr.Column():
                    vd_out = gr.Audio(label='🔊 Voice Design Output', type='numpy')
                    vd_status = gr.Textbox(label='Status', interactive=False)
                    gr.Markdown('**Presets:** Calm / Excited / News Anchor / Story — quick click!')

            p_calm.click(   fn=lambda: (0.8,  -2, 0.7, 0.3, 'speak calmly and peacefully'),                           outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            p_excited.click(fn=lambda: (1.3,   3, 1.5, 0.0, 'speak with excitement and high energy'),                 outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            p_news.click(   fn=lambda: (1.0,   0, 1.1, 0.2, 'speak like a professional news anchor, clear, formal'),  outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            p_story.click(  fn=lambda: (0.85, -1, 0.8, 0.4, 'speak like a storyteller, warm and engaging'),           outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            vd_btn.click(gen_voice_design, [vd_text,vd_lang,vd_speed,vd_pitch,vd_energy,vd_pause,vd_style], [vd_out,vd_status])

        # TAB 3: Simple TTS
        with gr.TabItem('🔤 Simple TTS'):
            with gr.Row():
                with gr.Column():
                    tts_text = gr.Textbox(label='📝 Text', lines=6, elem_classes='shiv-textbox', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b3 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b3.click(fn=None, inputs=[b3, tts_text], outputs=tts_text, js=INSERT_TAG_JS)
                    tts_lang     = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    tts_steps    = gr.Slider(label='🔢 Steps (Quality)', minimum=10, maximum=64, value=32, step=2)
                    tts_guidance = gr.Slider(label='🎯 Guidance Scale',  minimum=1.0, maximum=5.0, value=2.0, step=0.5)
                    tts_btn = gr.Button('🔤 Generate TTS', variant='primary', size='lg')
                with gr.Column():
                    tts_out    = gr.Audio(label='🔊 TTS Output', type='numpy')
                    tts_status = gr.Textbox(label='Status', interactive=False)
            tts_btn.click(gen_tts, [tts_text, tts_lang, tts_steps, tts_guidance], [tts_out, tts_status])

        # TAB 4: Instruct Mode
        with gr.TabItem('📋 Instruct Mode'):
            with gr.Row():
                with gr.Column():
                    inst_text = gr.Textbox(label='📝 Text', lines=5, elem_classes='shiv-textbox', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b4 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b4.click(fn=None, inputs=[b4, inst_text], outputs=inst_text, js=INSERT_TAG_JS)
                    inst_lang   = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    inst_prompt = gr.Textbox(label='📋 Style Instruction', lines=3, placeholder='Speak slowly and clearly...')
                    gr.Markdown('**💡 Examples (click karein):**')
                    for ex in INSTRUCT_EXAMPLES:
                        eb = gr.Button(ex, size='sm')
                        eb.click(fn=lambda x=ex: x, outputs=inst_prompt)
                    inst_btn = gr.Button('📋 Generate with Instruct', variant='primary', size='lg')
                with gr.Column():
                    inst_out    = gr.Audio(label='🔊 Instruct Output', type='numpy')
                    inst_status = gr.Textbox(label='Status', interactive=False)
            inst_btn.click(gen_instruct, [inst_text, inst_lang, inst_prompt], [inst_out, inst_status])

    gr.HTML("<div style='text-align:center;padding:15px;color:#888;'>© 2026 🔱 Shiv AI Voice Cloning | Shri Ram Nag | PAISAWALA</div>")

demo.launch(share=True, debug=False)
print('🔱 Shiv AI launched! Upar wala public URL copy karein.')